In [ ]:
# Portable project paths. Set TLS_PROJECT_ROOT to the directory containing the input data.
import os
from pathlib import Path
PROJECT_ROOT = Path(os.environ.get("TLS_PROJECT_ROOT", ".")).resolve()


# Supplementary Figure 35 plotting code


## Shared setup


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon, Rectangle, Circle, FancyArrowPatch
from scipy.spatial import ConvexHull, QhullError

ROOT = Path.cwd()
DATA = ROOT / "source_data"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)
assert DATA.exists(), "Run this notebook from the Figure 6 code directory."

mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "font.family": "DejaVu Sans",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "legend.frameon": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

blue = "#2B5C96"
red = "#C94C4C"
grey = "#8B9AA7"
dark = "#152536"

cell_colors = {
    "Epithelial": "#1F7A7A", "Fibroblast": "#E59B3A", "Endothelial": "#39A56A",
    "Pericyte": "#93B5D7", "T cell": "#E64B35", "NK cell": "#7E57C2",
    "B cell": "#2C7FB8", "Plasma cell": "#4EA3D8", "Monocyte/Macrophage": "#61C2A2",
    "Dendritic cell": "#D94F9B", "Mast cell": "#8CD36B", "Neutrophil": "#37BDBD",
    "Mix": "#C9C9C9", "Unknown": "#D9D9D9",
    "T/NK cell": "#4FA366", "Myeloid": "#EF6C42", "Fibroblast/FDC": "#F1A55C",
    "Endothelial/Pericyte": "#5EB4A5", "Other/Unknown": "#BDBDBD", "Other immune": "#BDBDBD",
    "B cells": "#2C7FB8", "DC": "#D94F9B", "DC cells": "#D94F9B",
    "Endothelial cells": "#39A56A", "Epithelial cells": "#1F7A7A",
    "Fibroblasts": "#E59B3A", "Macrophages": "#61C2A2",
    "Mast cells": "#8CD36B", "Monocytes": "#61C2A2",
    "T cells": "#E64B35", "NK cells": "#7E57C2"
}

class_colors = {"Conforming TLS": blue, "Deviating TLS": red, "Mature TLS": "#2B7A3D"}

from matplotlib.gridspec import GridSpec
def savefig(fig, name):
    for ext in ["pdf", "svg", "png"]:
        fig.savefig(OUT / f"{name}.{ext}", dpi=450, bbox_inches="tight")

def sem(x):
    x = pd.Series(x).dropna()
    return x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan

def draw_hull(ax, df, x="X", y="Y", color="k", lw=0.55, alpha=1, pad=0):
    pts = df[[x, y]].dropna().drop_duplicates().to_numpy()
    if len(pts) < 3:
        return
    try:
        h = ConvexHull(pts)
        poly = Polygon(pts[h.vertices], closed=True, fill=False, ec=color, lw=lw, alpha=alpha, joinstyle="round")
        ax.add_patch(poly)
    except QhullError:
        pass

def add_panel(ax, label):
    ax.text(-0.08, 1.05, label, transform=ax.transAxes, ha="left", va="bottom", fontsize=9, fontweight="bold")


## UMAP panel helper


In [ ]:
def umap_panel(ax, df, x, y, ct_col, title, label_centers=False):
    order = [c for c in cell_colors if c in set(df[ct_col])]
    for ct in order:
        sub = df[df[ct_col] == ct]
        ax.scatter(sub[x], sub[y], s=0.7, c=cell_colors[ct], lw=0, rasterized=True, label=ct)
    if label_centers:
        for ct in order:
            sub = df[df[ct_col] == ct]
            ax.text(sub[x].median(), sub[y].median(), ct, ha="center", va="center", fontsize=7, fontweight="bold",
                    bbox=dict(fc="white", ec="none", alpha=0.75, pad=1.2))
    ax.set_title(title, loc="left"); ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(markerscale=6, fontsize=6, bbox_to_anchor=(1.02, 0.5), loc="center left")


## Dot-plot helper


In [ ]:
def dotplot(ax, df, mean_col, pct_col, title):
    y_order = df["celltype"].drop_duplicates().tolist()
    x_order = df[["marker_group", "gene"]].drop_duplicates()["gene"].tolist()
    d = df.set_index(["celltype", "gene"])
    mx = max(float(df[mean_col].max()), 1e-9)
    for yi, ct in enumerate(y_order):
        for xi, gene in enumerate(x_order):
            if (ct, gene) in d.index:
                r = d.loc[(ct, gene)]
                if isinstance(r, pd.DataFrame): r = r.iloc[0]
                ax.scatter(xi, yi, s=max(r[pct_col], 1)*2.0, c=[plt.cm.YlOrRd(float(r[mean_col])/mx)], ec="black", lw=0.25)
    ax.set_xticks(range(len(x_order))); ax.set_xticklabels(x_order, rotation=50, ha="right")
    ax.set_yticks(range(len(y_order))); ax.set_yticklabels(y_order)
    ax.invert_yaxis(); ax.set_title(title, loc="left")
    groups = df[["marker_group", "gene"]].drop_duplicates().groupby("marker_group")["gene"].apply(list)
    pos = 0
    for group, genes in groups.items():
        center = pos + (len(genes)-1)/2
        ax.text(center, -1.10, group, ha="left", va="bottom", fontsize=5.5, rotation=35)
        if pos > 0:
            ax.axvline(pos-0.5, color="#D8E0E7", lw=0.8)
        pos += len(genes)
    norm = mpl.colors.Normalize(vmin=0, vmax=mx)
    sm = mpl.cm.ScalarMappable(cmap="YlOrRd", norm=norm)
    cb = plt.colorbar(sm, ax=ax, fraction=0.025, pad=0.02)
    cb.set_label("Mean expression")
    for pct in [25, 50, 75]:
        ax.scatter([], [], s=pct*2.0, c="white", ec="black", lw=0.25, label=f"{pct}%")
    ax.legend(title="Expressing", fontsize=5, title_fontsize=6, bbox_to_anchor=(1.18, 0.05), loc="lower left")
